In [ ]:
# Simulate DAB variable interpolation by passing variables via args parameter

# Use this script to test a create table script under /sql/dml/

# Change the load_sql parameters to match your environment
catalog_name = "dev_catalog"
schema_prefix = "slv_cdm_hrs"
#schema_prefix = "gld_star_hrs"

#load_sql = "dml/load_hrs_cohort_data.sql"
load_sql = "dml/load_hrs_wave_data.sql"
#load_sql = "dml/load_hrs_respondent_data.sql"
#load_sql = "dml/load_hrs_demographics_data.sql"
#load_sql = "dml/load_hrs_health_data.sql"
#load_sql = "dml/load_hrs_leave_behind_data.sql"
#load_sql = "gold_dml/load_fact_hrs_bmi_stats_data.sql"
#load_sql = "gold_dml/load_fact_hrs_bmi_race_gender_stats.sql"


# Read the SQL file
with open(f"/Workspace/Users/peteperez.lv@gmail.com/hrs_dbx_repo/sql/{load_sql}", "r") as f:
    sql_content = f.read()

# remove comments from the sql statement to avoid sql statement splitting.
sql_content = "\n".join( line for line in sql_content.splitlines() if not line.strip().startswith("--"))

# Split SQL statements by semicolon and execute each separately
statements = [
    stmt.strip()
    for stmt in sql_content.split(";")
    if stmt.strip()
]

target_table = f"{catalog_name}.{schema_prefix}.dim_cohort"

for stmt in statements:
    # Check if this is an INSERT statement
    if stmt.upper().strip().startswith("INSERT"):
        # Execute the SELECT portion to get a DataFrame, then write using DataFrame API
        # Extract the SELECT portion after the closing parenthesis of the column list
        select_start = stmt.upper().find("SELECT")
        select_query = stmt[select_start:]
        
        # Execute SELECT with parameters
        df = spark.sql(select_query, args={"catalog_name": catalog_name, "schema_prefix": schema_prefix})
        
        # Write using DataFrame API instead of SQL INSERT
        df.write.mode("append").saveAsTable(target_table)
    else:
        # For non-INSERT statements (TRUNCATE, etc.), use SQL directly
        spark.sql(stmt, args={"catalog_name": catalog_name, "schema_prefix": schema_prefix})

%md
## How This Simulates DAB Variables

This notebook simulates how Declarative Automation Bundle (DAB) variable interpolation works:

1. **Variables defined**: `catalog_name` and `schema_prefix`
2. **SQL file uses**: `:catalog_name` and `:schema_prefix` parameter syntax
3. **At runtime**: The `args` parameter in `spark.sql()` binds the values

In a deployed DAB job, the bundle would interpolate `${catalog_name}` at deployment time. This approach simulates that behavior for local testing.